In [1]:
# ============================================
# FILE: towuti_phage_pipeline.ipynb
# ============================================

In [2]:
# ============================================
# MODULE 0 — PROJECT SETUP
# ============================================

import os
from pathlib import Path

PROJECT_DIR = Path.cwd()

dirs = [
    "data/raw",
    "data/clean",
    "data/subsampled",
    "data/assembly",
    "qc/raw_fastqc",
    "qc/clean_fastqc",
    "qc/multiqc",
    "results/contigs",
    "results/phabox",
    "results/figures",
    "logs",
]

for d in dirs:
    Path(d).mkdir(parents=True, exist_ok=True)

print("Project directories created.")

Project directories created.


In [3]:
# ============================================
# MODULE 1 — CEK ENVIRONMENT
# ============================================

!python --version
!fastp --version
!megahit --version
!seqkit version

Python 3.11.15
fastp 1.3.3
MEGAHIT v1.2.9
seqkit v2.13.0


In [ ]:
# ============================================
# MODULE 2,1 — DOWNLOAD FASTQ 1 FROM ENA
# ============================================

!mkdir -p /tmp/raw
!wget -O /tmp/raw/SRR8436514_1.fastq.gz "http://ftp.sra.ebi.ac.uk/vol1/fastq/SRR843/004/SRR8436514/SRR8436514_1.fastq.gz"

--2026-05-18 16:08:25--  http://ftp.sra.ebi.ac.uk/vol1/fastq/SRR843/004/SRR8436514/SRR8436514_1.fastq.gz
Resolving ftp.sra.ebi.ac.uk (ftp.sra.ebi.ac.uk)... 193.62.193.165
Connecting to ftp.sra.ebi.ac.uk (ftp.sra.ebi.ac.uk)|193.62.193.165|:80... 

In [6]:
# ============================================
# MODULE 2.2 — DOWNLOAD FASTQ 2 FROM ENA
# ============================================

!mkdir -p /tmp/raw
!wget -O /tmp/raw/SRR8436514_2.fastq.gz "http://ftp.sra.ebi.ac.uk/vol1/fastq/SRR843/004/SRR8436514/SRR8436514_2.fastq.gz"

--2026-05-18 15:31:35--  http://ftp.sra.ebi.ac.uk/vol1/fastq/SRR843/004/SRR8436514/SRR8436514_2.fastq.gz
Resolving ftp.sra.ebi.ac.uk (ftp.sra.ebi.ac.uk)... 193.62.193.165
Connecting to ftp.sra.ebi.ac.uk (ftp.sra.ebi.ac.uk)|193.62.193.165|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10026339650 (9.3G) [application/x-gzip]
Saving to: ‘/tmp/raw/SRR8436514_2.fastq.gz’

/tmp/raw/SRR8436514 100%[===================>]   9.34G  3.43MB/s    in 35m 37s 

2026-05-18 16:07:14 (4.47 MB/s) - ‘/tmp/raw/SRR8436514_2.fastq.gz’ saved [10026339650/10026339650]



In [ ]:
# ============================================
# MODULE 3 — SKIP SRA CONVERSION
# ============================================

"""
FASTQ files telah diunduh ke /tmp:
- /tmp/raw/SRR8436514_1.fastq.gz
- /tmp/raw/SRR8436514_2.fastq.gz

Lanjutkan langsung ke quality control.
"""

In [7]:
# ============================================
# MODULE 4 — CEK FILE RAW
# ============================================

!ls -lh /tmp/raw

total 9.4G
-rw-r--rw- 1 codespace codespace 9.4G Jan 14  2019 SRR8436514_2.fastq.gz


In [8]:
# ============================================
# MODULE 5 — FASTQC RAW READS
# ============================================

!fastqc \
/tmp/raw/SRR8436514_1.fastq.gz \
/tmp/raw/SRR8436514_2.fastq.gz \
-o qc/raw_fastqc

Skipping '/tmp/raw/SRR8436514_1.fastq.gz' which didn't exist, or couldn't be read
application/gzip
Started analysis of SRR8436514_2.fastq.gz
^C


In [ ]:
# ============================================
# MODULE 6 — TRIMMING + FILTERING
# ============================================

!fastp \
-i /tmp/raw/SRR8436514_1.fastq.gz \
-I /tmp/raw/SRR8436514_2.fastq.gz \
-o data/clean/clean_1.fastq \
-O data/clean/clean_2.fastq \
-h qc/fastp.html \
-j qc/fastp.json \
-w 2

In [ ]:
# ============================================
# MODULE 7 — FASTQC CLEAN READS
# ============================================

!fastqc \
data/clean/clean_1.fastq \
data/clean/clean_2.fastq \
-o qc/clean_fastqc

In [ ]:
# ============================================
# MODULE 8 — MULTIQC
# ============================================

!multiqc qc \
-o qc/multiqc

In [ ]:
# ============================================
# MODULE 9 — SUBSAMPLING (25%)
# ============================================

!seqkit sample \
-p 0.25 \
-s 123 \
data/clean/clean_1.fastq \
> data/subsampled/sub_1.fastq

!seqkit sample \
-p 0.25 \
-s 123 \
data/clean/clean_2.fastq \
> data/subsampled/sub_2.fastq

In [ ]:
# ============================================
# MODULE 10 — CEK SUBSAMPLE
# ============================================

!seqkit stats data/subsampled/sub_1.fastq
!seqkit stats data/subsampled/sub_2.fastq

In [ ]:
# ============================================
# MODULE 11 — METAGENOME ASSEMBLY
# ============================================

!megahit \
-1 data/subsampled/sub_1.fastq \
-2 data/subsampled/sub_2.fastq \
-o data/assembly/megahit_out \
-t 2 \
--min-contig-len 1000

In [ ]:
# ============================================
# MODULE 12 — CEK HASIL ASSEMBLY
# ============================================

!seqkit stats \
data/assembly/megahit_out/final.contigs.fa

In [ ]:
# ============================================
# MODULE 13 — HITUNG CONTIGS
# ============================================

!grep ">" \
data/assembly/megahit_out/final.contigs.fa \
| wc -l

In [ ]:
# ============================================
# MODULE 14 — FILTER CONTIG >3kb
# ============================================

!seqkit seq \
-m 3000 \
data/assembly/megahit_out/final.contigs.fa \
> results/contigs/contigs_3kb.fa

In [ ]:
# ============================================
# MODULE 15 — STATISTIK CONTIG FINAL
# ============================================

!seqkit stats \
results/contigs/contigs_3kb.fa

In [ ]:
# ============================================
# MODULE 16 — KOMPRES OUTPUT
# ============================================

!gzip -f results/contigs/contigs_3kb.fa

In [ ]:
# ============================================
# MODULE 17 — CEK OUTPUT FINAL
# ============================================

!ls -lh results/contigs

In [ ]:
# ============================================
# MODULE 18 — NEXT STEP
# ============================================

"""
UPLOAD:
results/contigs/contigs_3kb.fa.gz

TO:
https://phage.ee.cityu.edu.hk/phabox

SELECT:
- Virus identification
- Taxonomy
- Host prediction

OPTIONAL:
- CheckV
- iPHoP
- vConTACT2
"""